# 00 — Python Environment QC and Reproducibility Freeze

The preprocessing notebooks showed a warning of the form:

`numpy.ndarray size changed, may indicate binary incompatibility`

This notebook isolates that problem from the scientific analysis.

## What this notebook does

1. Launches a **fresh Python subprocess** and imports the compiled scientific stack.
2. Flags NumPy ABI / binary-compatibility warnings.
3. Runs small read/write and numerical smoke tests.
4. Saves:
   - Python/package versions;
   - `pip freeze`;
   - `conda list --explicit`;
   - `conda env export --no-builds`;
   - the complete subprocess stderr/stdout.
5. Writes clean-environment rebuild commands for Windows/Conda.

## Important

This notebook **does not modify or reinstall packages automatically**.
Environment mutation should be done explicitly in Anaconda Prompt.

You do **not** need to rerun the 257+ GB preprocessing merely because the old
environment emitted a warning. First create/verify a clean environment, then
use it for reproducibility and future reruns/QC.


In [1]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import tempfile
import warnings

BASE = Path(r"D:\ERA5_LAND")

OUTPUT_DIR = (
    BASE
    / "environment_snapshot"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Python executable:", sys.executable)
print("Python version   :", sys.version)
print("Platform         :", platform.platform())


Python executable: c:\Users\User\anaconda3\envs\era5\python.exe
Python version   : 3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)]
Platform         : Windows-11-10.0.26200-SP0


In [2]:
# ============================================================
# Fresh-subprocess ABI / import diagnostic
# ============================================================
smoke_script = r'''
import warnings
warnings.simplefilter("always")

packages = [
    "numpy",
    "pandas",
    "scipy",
    "xarray",
    "dask",
    "netCDF4",
    "h5py",
    "h5netcdf",
    "bottleneck",
    "numba",
    "matplotlib",
]

for name in packages:
    mod = __import__(name)
    print(
        f"{name}={getattr(mod, '__version__', 'unknown')}"
    )
'''

proc = subprocess.run(
    [
        sys.executable,
        "-W",
        "always",
        "-c",
        smoke_script,
    ],
    capture_output=True,
    text=True,
)

stdout = proc.stdout
stderr = proc.stderr

print("Fresh subprocess return code:", proc.returncode)
print("\nVERSIONS")
print(stdout)

if stderr.strip():
    print("\nSUBPROCESS STDERR / WARNINGS")
    print(stderr)

binary_warning_patterns = [
    "numpy.ndarray size changed",
    "binary incompatibility",
    "dtype size changed",
    "module compiled against",
]

binary_warning_detected = any(
    pattern.lower() in stderr.lower()
    for pattern in binary_warning_patterns
)

print(
    "\nBinary-compatibility warning detected:",
    binary_warning_detected,
)

(OUTPUT_DIR / "fresh_import_stdout.txt").write_text(
    stdout,
    encoding="utf-8",
)

(OUTPUT_DIR / "fresh_import_stderr.txt").write_text(
    stderr,
    encoding="utf-8",
)


Fresh subprocess return code: 0

VERSIONS
numpy=2.4.6
pandas=3.0.5
scipy=1.18.0
xarray=2026.7.0
dask=2026.7.1
netCDF4=1.7.4
h5py=3.16.0
h5netcdf=1.8.1
bottleneck=1.4.2
numba=0.66.0
matplotlib=3.11.0


Binary-compatibility warning detected: False


0

In [3]:
# ============================================================
# In-process package versions
# ============================================================
packages = [
    "numpy",
    "pandas",
    "scipy",
    "xarray",
    "dask",
    "netCDF4",
    "h5py",
    "h5netcdf",
    "bottleneck",
    "numba",
    "matplotlib",
]

versions = {
    "python": sys.version.split()[0],
    "python_executable": sys.executable,
    "platform": platform.platform(),
}

for name in packages:
    try:
        mod = __import__(name)
        versions[name] = getattr(
            mod,
            "__version__",
            "unknown",
        )
    except Exception as exc:
        versions[name] = (
            f"IMPORT ERROR: "
            f"{type(exc).__name__}: {exc}"
        )

versions["binary_warning_detected_fresh_subprocess"] = (
    binary_warning_detected
)

(OUTPUT_DIR / "package_versions.json").write_text(
    json.dumps(
        versions,
        indent=2,
    ),
    encoding="utf-8",
)

versions


{'python': '3.12.13',
 'python_executable': 'c:\\Users\\User\\anaconda3\\envs\\era5\\python.exe',
 'platform': 'Windows-11-10.0.26200-SP0',
 'numpy': '2.4.6',
 'pandas': '3.0.5',
 'scipy': '1.18.0',
 'xarray': '2026.7.0',
 'dask': '2026.7.1',
 'netCDF4': '1.7.4',
 'h5py': '3.16.0',
 'h5netcdf': '1.8.1',
 'bottleneck': '1.4.2',
 'numba': '0.66.0',
 'matplotlib': '3.11.0',
 'binary_warning_detected_fresh_subprocess': False}

In [4]:
# ============================================================
# Small numerical / NetCDF smoke tests
# ============================================================
import numpy as np
import xarray as xr

smoke_results = {}

# Numerical sanity.
a = np.arange(
    24,
    dtype=np.float64,
).reshape(2, 3, 4)

smoke_results["numpy_mean_ok"] = bool(
    np.isclose(
        a.mean(),
        11.5,
    )
)

# xarray operation.
da = xr.DataArray(
    a,
    dims=("x", "y", "z"),
)

smoke_results["xarray_mean_ok"] = bool(
    np.isclose(
        float(da.mean()),
        11.5,
    )
)

# NetCDF read/write tests using both engines where available.
test_ds = xr.Dataset({
    "a": (
        ("x", "y"),
        np.arange(12, dtype=np.float32).reshape(3, 4),
    )
})

for engine in [
    "h5netcdf",
    "netcdf4",
]:
    try:
        test_file = (
            OUTPUT_DIR
            / f"_environment_smoke_{engine}.nc"
        )

        test_ds.to_netcdf(
            test_file,
            engine=engine,
        )

        reread = xr.open_dataset(
            test_file,
            engine=engine,
        )

        ok = np.array_equal(
            reread["a"].values,
            test_ds["a"].values,
        )

        reread.close()
        test_file.unlink(
            missing_ok=True
        )

        smoke_results[
            f"{engine}_roundtrip_ok"
        ] = bool(ok)

    except Exception as exc:
        smoke_results[
            f"{engine}_roundtrip_ok"
        ] = False
        smoke_results[
            f"{engine}_error"
        ] = (
            f"{type(exc).__name__}: {exc}"
        )

# Numba compilation.
try:
    from numba import njit

    @njit
    def _numba_sum(x):
        total = 0.0
        for v in x:
            total += v
        return total

    numba_value = _numba_sum(
        np.arange(10, dtype=np.float64)
    )

    smoke_results["numba_compile_ok"] = bool(
        np.isclose(
            numba_value,
            45.0,
        )
    )

except Exception as exc:
    smoke_results["numba_compile_ok"] = False
    smoke_results["numba_error"] = (
        f"{type(exc).__name__}: {exc}"
    )

smoke_results["all_smoke_tests_pass"] = bool(
    all(
        value
        for key, value in smoke_results.items()
        if key.endswith("_ok")
    )
)

(OUTPUT_DIR / "environment_smoke_tests.json").write_text(
    json.dumps(
        smoke_results,
        indent=2,
    ),
    encoding="utf-8",
)

smoke_results


{'numpy_mean_ok': True,
 'xarray_mean_ok': True,
 'h5netcdf_roundtrip_ok': True,
 'netcdf4_roundtrip_ok': True,
 'numba_compile_ok': True,
 'all_smoke_tests_pass': True}

In [5]:
# ============================================================
# Freeze the current environment for traceability
# ============================================================
def run_and_save(
    args,
    filename,
):
    try:
        result = subprocess.run(
            args,
            capture_output=True,
            text=True,
            check=False,
        )

        text = result.stdout

        if result.stderr.strip():
            text += (
                "\n\n# STDERR\n"
                + result.stderr
            )

        (
            OUTPUT_DIR
            / filename
        ).write_text(
            text,
            encoding="utf-8",
        )

        return result.returncode

    except Exception as exc:
        (
            OUTPUT_DIR
            / filename
        ).write_text(
            (
                f"COMMAND FAILED: "
                f"{type(exc).__name__}: {exc}\n"
            ),
            encoding="utf-8",
        )
        return -1


freeze_status = {}

freeze_status["pip_freeze"] = run_and_save(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze",
    ],
    "pip_freeze.txt",
)

freeze_status["conda_list_explicit"] = run_and_save(
    [
        "conda",
        "list",
        "--explicit",
    ],
    "conda_list_explicit.txt",
)

freeze_status["conda_env_export"] = run_and_save(
    [
        "conda",
        "env",
        "export",
        "--no-builds",
    ],
    "conda_environment_no_builds.yml",
)

freeze_status


{'pip_freeze': 0, 'conda_list_explicit': -1, 'conda_env_export': -1}

In [6]:
# ============================================================
# Write safe rebuild commands.
# ============================================================
rebuild_commands = r'''
# Run in Anaconda Prompt, NOT inside the notebook.

conda create -n era5_clean -c conda-forge ^
  python=3.12 ^
  numpy pandas scipy xarray dask ^
  netcdf4 h5py h5netcdf bottleneck ^
  numba matplotlib ^
  cartopy geopandas shapely ^
  jupyterlab ipykernel -y

conda activate era5_clean

python -m ipykernel install --user ^
  --name era5_clean ^
  --display-name "Python (era5_clean)"

# Then reopen this QC notebook using the new kernel.
# Only when all import/smoke tests pass, freeze the clean environment:

conda list --explicit > D:\ERA5_LAND\environment_snapshot\era5_clean_conda_explicit.txt
conda env export --no-builds > D:\ERA5_LAND\environment_snapshot\era5_clean_environment.yml
python -m pip freeze > D:\ERA5_LAND\environment_snapshot\era5_clean_pip_freeze.txt
'''

(
    OUTPUT_DIR
    / "REBUILD_ENVIRONMENT_WINDOWS.txt"
).write_text(
    rebuild_commands.strip()
    + "\n",
    encoding="utf-8",
)

print(rebuild_commands)



# Run in Anaconda Prompt, NOT inside the notebook.

conda create -n era5_clean -c conda-forge ^
  python=3.12 ^
  numpy pandas scipy xarray dask ^
  netcdf4 h5py h5netcdf bottleneck ^
  numba matplotlib ^
  cartopy geopandas shapely ^
  jupyterlab ipykernel -y

conda activate era5_clean

python -m ipykernel install --user ^
  --name era5_clean ^
  --display-name "Python (era5_clean)"

# Then reopen this QC notebook using the new kernel.
# Only when all import/smoke tests pass, freeze the clean environment:

conda list --explicit > D:\ERA5_LAND\environment_snapshot\era5_clean_conda_explicit.txt
conda env export --no-builds > D:\ERA5_LAND\environment_snapshot\era5_clean_environment.yml
python -m pip freeze > D:\ERA5_LAND\environment_snapshot\era5_clean_pip_freeze.txt



In [7]:
# ============================================================
# Final environment verdict
# ============================================================
environment_pass = bool(
    proc.returncode == 0
    and not binary_warning_detected
    and smoke_results[
        "all_smoke_tests_pass"
    ]
)

verdict = {
    "fresh_import_return_code": proc.returncode,
    "binary_warning_detected": binary_warning_detected,
    "all_smoke_tests_pass": smoke_results[
        "all_smoke_tests_pass"
    ],
    "environment_pass": environment_pass,
    "action": (
        "environment is suitable for reproducible future reruns"
        if environment_pass
        else (
            "create a clean conda-forge environment, rerun this QC, "
            "and freeze only after the clean environment passes"
        )
    ),
}

(
    OUTPUT_DIR
    / "environment_verdict.json"
).write_text(
    json.dumps(
        verdict,
        indent=2,
    ),
    encoding="utf-8",
)

print(json.dumps(verdict, indent=2))

if not environment_pass:
    print(
        "\nIMPORTANT: This does not automatically invalidate "
        "existing ERA5-Land outputs. It means the software environment "
        "should be repaired/frozen before future reproducibility runs."
    )


{
  "fresh_import_return_code": 0,
  "binary_warning_detected": false,
  "all_smoke_tests_pass": true,
  "environment_pass": true,
  "action": "environment is suitable for reproducible future reruns"
}
